In [1]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd
import torch
from collections import defaultdict, Counter
from sentence_transformers import SentenceTransformer
import xml.dom.minidom
import string
from tqdm import tqdm
from sklearn.cluster import KMeans, MiniBatchKMeans, Birch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import f1_score
import matplotlib.pyplot as plt

/root/anaconda3/envs/tsw/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
questionaire_single = [ #   BDI-II 抑郁症测量表定义的症状----抑郁症模板中的第二组
    "I feel sad.",
    "I am discouraged about my future.",
    "I always fail.",
    "I don't get pleasure from things.",
    "I feel quite guilty.",
    "I expected to be punished.",
    "I am disappointed in myself.",
    "I always criticize myself for my faults.",
    "I have thoughts of killing myself.",
    "I always cry.",
    "I am hard to stay still.",
    "It's hard to get interested in things.",
    "I have trouble making decisions.",
    "I feel worthless.",
    "I don't have energy to do things.",
    "I have changes in my sleeping pattern.",
    "I am always irritable.",
    "I have changes in my appetite.",
    "I feel hard to concentrate on things.",
    "I am too tired to do things.",
    "I have lost my interest in sex."
]
print(len(questionaire_single))

21


In [3]:
depression_texts = [     #抑郁症模板中的第一组：3个显性抑郁的表达组成，对应患者的抑郁情况。
    "I feel depressed.",
    "I am diagnosed with depression.",
    "I am treating my depression."
]

In [4]:
with open("/E22301339/HAN-BERT/eRisk2017/processed/miniLM_L6_embs.pkl", "rb") as f:
    data = pickle.load(f)       # 读取指定的二进制对象，并返回序列化对象

train_posts = data["train_posts"]
train_mappings = data["train_mappings"]
train_tags = data["train_labels"]
train_embs = data["train_embs"]
test_posts = data["test_posts"]
test_mappings = data["test_mappings"]
test_tags = data["test_labels"]
test_embs = data["test_embs"]
# print(train_embs.size,train_embs.shape,60067*384)
# print(train_embs[0],type(train_embs[0]),train_embs[0].size)

In [5]:
sbert = SentenceTransformer('/E22301339/paraphrase-MiniLM-L6-v2')   #sentence-transformers模型

No sentence-transformers model found with name /E22301339/paraphrase-MiniLM-L6-v2. Creating a new one with MEAN pooling.


In [6]:
questionaire_single_embs = sbert.encode(questionaire_single)  #计算表示
depression_embs = sbert.encode(depression_texts)

In [7]:
# take care, require ~100G RAM
train_posts = np.array(train_posts)   #将python列表转换为numpy数组
test_posts = np.array(test_posts)

In [8]:
depression_pair_sim = cosine_similarity(train_embs, depression_embs)  # 根据训练集中的帖子和抑郁症三个显性表达的嵌入计算余弦相似度
depression_pair_sim.shape
print(type(depression_pair_sim))

<class 'numpy.ndarray'>


In [9]:
depression_pair_sim_test = cosine_similarity(test_embs, depression_embs)# 根据测试集中的帖子和抑郁症三个显性表达的嵌入计算余弦相似度
depression_pair_sim_test.shape                                          # 该相似度也被视为该帖子的风险

(236371, 3)

In [ ]:
#   获取K=16个最高风险的帖子：利用帖子与模板第一组计算的相似度
topK = 16
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/depress_sim{topK}", exist_ok=True)  #创建文件夹；若已存在则不创建
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/depress_sim{topK}/train", exist_ok=True)
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/depress_sim{topK}/test", exist_ok=True)
for i, (mapping, label) in enumerate(zip(train_mappings, train_tags)):
    posts = train_posts[mapping]
    sim_scores = depression_pair_sim[mapping, 0] #取与第一组模板中的"I feel depressed.的余弦相似度得分
    top_ids = sim_scores.argsort()[-topK:]                                                      
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    with open(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/depress_sim{topK}/train/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

for i, (mapping, label) in enumerate(zip(test_mappings, test_tags)):
    posts = test_posts[mapping]
    sim_scores = depression_pair_sim_test[mapping, 0]
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    with open(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/depress_sim{topK}/test/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

In [10]:
dimension_sim_single = cosine_similarity(train_embs, questionaire_single_embs)  #计算训练集中的帖子与抑郁症模板第二组之间的余弦相似度
dimension_sim_single.shape

(295023, 21)

In [11]:
dimension_sim_single_test = cosine_similarity(test_embs, questionaire_single_embs)#计算测试集中的帖子与抑郁症模板第二组之间的余弦相似度
dimension_sim_single_test.shape

(236371, 21)

In [12]:
# 在第二个维度（列）上进行拼接 连接帖子与模板第一组和模板第二组之间的余弦相似度,在训练集上
combined_sim = np.concatenate([depression_pair_sim, dimension_sim_single], axis=1)  
combined_sim_test = np.concatenate([depression_pair_sim_test, dimension_sim_single_test], axis=1)
combined_sim.shape, combined_sim_test.shape

((295023, 24), (236371, 24))

In [ ]:
#   获取K=16个最高风险的帖子：利用上面连接后的向量计算比较
topK = 16
os.makedirs(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK}", exist_ok=True)
os.makedirs(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK}/train", exist_ok=True)
os.makedirs(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK}/test", exist_ok=True)
for i, (mapping, label) in enumerate(zip(train_mappings, train_tags)):
    posts = train_posts[mapping]
    sim_scores = combined_sim[mapping].max(1)   #axis=1 表示我们要沿着第二个轴的方向进行求解max。这就是 combined_sim[mapping].max(1) 这行代码的含义。
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    with open(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK}/train/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

for i, (mapping, label) in enumerate(zip(test_mappings, test_tags)):
    posts = test_posts[mapping]
    sim_scores = combined_sim_test[mapping].max(1)
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    with open(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK}/test/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

In [ ]:
#   获取K=16个最高风险的帖子：利用帖子与模板第二组计算的相似度
topK = 16
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/questionaire_maxsim{topK}", exist_ok=True)
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/questionaire_maxsim{topK}/train", exist_ok=True)
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/questionaire_maxsim{topK}/test", exist_ok=True)
for i, (mapping, label) in enumerate(zip(train_mappings, train_tags)):
    posts = train_posts[mapping]
    sim_scores = dimension_sim_single[mapping].max(1)
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    with open(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/questionaire_maxsim{topK}/train/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

for i, (mapping, label) in enumerate(zip(test_mappings, test_tags)):
    posts = test_posts[mapping]
    sim_scores = dimension_sim_single_test[mapping].max(1)
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    with open(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/questionaire_maxsim{topK}/test/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

In [ ]:
#   获取K=16个用户最近的帖子
'''
这段代码与前两段的不同在于，它只选择了每个用户的最后几篇文章（由topK决定），而不是根据文章之间的相似度进行筛选,因此可能更加高效。
'''
topK = 16
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/last{topK}", exist_ok=True)
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/last{topK}/train", exist_ok=True)
os.makedirs(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/last{topK}/test", exist_ok=True)
for i, (mapping, label) in enumerate(zip(train_mappings, train_tags)):
    posts = train_posts[mapping]
    sel_posts = posts[-topK:]
    with open(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/last{topK}/train/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

for i, (mapping, label) in enumerate(zip(test_mappings, test_tags)):
    posts = test_posts[mapping]
    sel_posts = posts[-topK:]
    with open(f"/home/E22301339/scale_early_depress_detect-main/eRisk2017/processed/last{topK}/test/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

#   K=16用户高风险帖子+K=16用户最近16条帖子

In [ ]:
#   获取K=16个最高风险的帖子：利用上面连接后的向量计算比较
topK = 16
topK2 = 4
os.makedirs(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK*2}", exist_ok=True)
os.makedirs(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK*2}/train", exist_ok=True)
os.makedirs(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK*2}/test", exist_ok=True)
for i, (mapping, label) in enumerate(zip(train_mappings, train_tags)):
    posts = train_posts[mapping]
    sim_scores = combined_sim[mapping].max(1)   #axis=1 表示我们要沿着第二个轴的方向进行求解max。这就是 combined_sim[mapping].max(1) 这行代码的含义。
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    sel_posts_late = posts[-topK2:]
    # sel_posts = np.concatenate((sel_posts, sel_posts_late), axis=None)
    sel_posts = np.union1d(sel_posts, sel_posts_late)

    with open(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK*2}/train/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

for i, (mapping, label) in enumerate(zip(test_mappings, test_tags)):
    posts = test_posts[mapping]
    sim_scores = combined_sim_test[mapping].max(1)
    top_ids = sim_scores.argsort()[-topK:]
    top_ids = np.sort(top_ids)  # sort in time order
    sel_posts = posts[top_ids]
    sel_posts_late = posts[-topK2:]
    sel_posts = np.concatenate((sel_posts, sel_posts_late), axis=None)
    with open(f"/E22301339/HAN-BERT/eRisk2017/processed/combined_maxsim{topK*2}/test/{i:06}_{label}.txt", "w") as f:
        f.write("\n".join(x.replace("\n", " ") for x in sel_posts))

